# 多渠道广告投放效果分析 — 数据准备与探索性分析

**项目背景**：模拟新能源汽车品牌（蔚来）多渠道广告投放场景，对投放数据进行清洗、特征工程和EDA，为后续Tableau可视化Dashboard提供干净数据。

**数据来源**：[Kaggle - Marketing Campaign Performance Dataset](https://www.kaggle.com/datasets/manishabhatt22/marketing-campaign-performance-dataset)

**工具**：Python 3.9 / Pandas / NumPy / Matplotlib / Seaborn

## 1. 数据加载与初步检查

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# macOS 中文字体配置 — 清除缓存 + 注册字体
mpl.font_manager._load_fontmanager(try_read_cache=False)
_font_path = '/System/Library/Fonts/STHeiti Medium.ttc'
fm.fontManager.addfont(_font_path)
_font_name = fm.FontProperties(fname=_font_path).get_name()

sns.set_style('whitegrid')
# 必须在 sns.set_style 之后设置，否则会被覆盖
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [_font_name] + mpl.rcParams['font.sans-serif']
mpl.rcParams['axes.unicode_minus'] = False
print(f'字体加载成功: {_font_name}')

# 加载数据
df = pd.read_csv('marketing_campaign_dataset.csv')
print(f'数据集形状: {df.shape}')
print(f'\n--- 数据类型 ---')
print(df.dtypes)
print(f'\n--- 前5行 ---')
df.head()

In [ ]:
# 描述性统计
df.describe(include='all')

In [ ]:
# 缺失值检查
missing = df.isnull().sum()
print('--- 缺失值统计 ---')
print(missing[missing > 0] if missing.sum() > 0 else '无缺失值')

# 重复值检查
dup_count = df.duplicated().sum()
print(f'\n重复行数: {dup_count}')
if dup_count > 0:
    df = df.drop_duplicates()
    print(f'已删除重复行，剩余: {len(df)} 行')

## 2. 数据清洗

In [ ]:
# 2.1 标准化列名（小写+下划线）
df.columns = df.columns.str.lower().str.replace(' ', '_')
print('标准化后的列名:', list(df.columns))

In [ ]:
# 2.2 清洗 acquisition_cost（去掉$和逗号，转为数值）
df['acquisition_cost'] = (
    df['acquisition_cost']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)
print('acquisition_cost 清洗完成')
print(df['acquisition_cost'].describe())

In [ ]:
# 2.3 日期列转换
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['weekday'] = df['date'].dt.day_name()

print(f'日期范围: {df["date"].min()} ~ {df["date"].max()}')

In [ ]:
# 2.4 类别字段检查与统一
cat_cols = ['campaign_type', 'channel_used', 'target_audience', 
            'location', 'language', 'customer_segment']

for col in cat_cols:
    df[col] = df[col].str.strip()
    print(f'{col}: {df[col].nunique()} 个唯一值 -> {df[col].unique()}')

In [ ]:
# 2.5 异常值处理
print('--- 异常值检查 ---')

# conversion_rate 应在 0~1 之间
invalid_cvr = (df['conversion_rate'] < 0) | (df['conversion_rate'] > 1)
print(f'conversion_rate 异常值: {invalid_cvr.sum()} 行')

# clicks 不应超过 impressions
invalid_clicks = df['clicks'] > df['impressions']
print(f'clicks > impressions 异常: {invalid_clicks.sum()} 行')

# 负数检查
for col in ['clicks', 'impressions', 'acquisition_cost']:
    neg = (df[col] < 0).sum()
    if neg > 0:
        print(f'{col} 存在 {neg} 个负值')

# ROI 极端值
print(f'\nROI 范围: {df["roi"].min():.2f} ~ {df["roi"].max():.2f}')

## 3. 特征工程 — 新增计算字段

In [ ]:
# 将 acquisition_cost 作为广告花费(spend)
df.rename(columns={'acquisition_cost': 'spend'}, inplace=True)

# CTR = clicks / impressions
df['ctr'] = df['clicks'] / df['impressions']

# 转化数 conversions = clicks * conversion_rate
df['conversions'] = (df['clicks'] * df['conversion_rate']).round(0).astype(int)

# CVR = conversions / clicks（即 conversion_rate，保留原始值作验证）
df['cvr'] = df['conversions'] / df['clicks']

# CPC = spend / clicks
df['cpc'] = df['spend'] / df['clicks']

# CPM = spend / impressions * 1000
df['cpm'] = df['spend'] / df['impressions'] * 1000

# Revenue = spend * (1 + ROI)，基于 ROI 定义: ROI = (revenue - spend) / spend
df['revenue'] = df['spend'] * (1 + df['roi'])

# ROAS = revenue / spend
df['roas'] = df['revenue'] / df['spend']

# CPA = spend / conversions（转化数为0时设为NaN）
df['cpa'] = df['spend'] / df['conversions'].replace(0, np.nan)

print('新增字段完成，当前列:')
print(list(df.columns))
print(f'\n--- 新字段统计 ---')
df[['ctr', 'cvr', 'cpc', 'cpm', 'roas', 'cpa']].describe()

## 4. 探索性数据分析（EDA）

### 4.1 各渠道投放效果对比

In [ ]:
# 各渠道核心指标汇总
channel_summary = df.groupby('channel_used').agg(
    总花费=('spend', 'sum'),
    总曝光=('impressions', 'sum'),
    总点击=('clicks', 'sum'),
    总转化=('conversions', 'sum'),
    平均CTR=('ctr', 'mean'),
    平均CVR=('cvr', 'mean'),
    平均CPC=('cpc', 'mean'),
    平均ROAS=('roas', 'mean')
).round(4)

channel_summary

In [ ]:
# 各渠道CTR、CVR、ROAS分布 — 箱线图
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric, title in zip(axes, ['ctr', 'cvr', 'roas'], 
                              ['点击率 CTR', '转化率 CVR', '广告回报率 ROAS']):
    sns.boxplot(data=df, x='channel_used', y=metric, ax=ax, palette='Set2')
    ax.set_title(f'各渠道{title}分布', fontsize=14)
    ax.set_xlabel('投放渠道')
    ax.set_ylabel(title)

plt.tight_layout()
plt.savefig('eda_channel_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 各广告类型效果对比

In [ ]:
# 各广告类型核心指标
type_summary = df.groupby('campaign_type').agg(
    总花费=('spend', 'sum'),
    总转化=('conversions', 'sum'),
    平均CTR=('ctr', 'mean'),
    平均CVR=('cvr', 'mean'),
    平均ROAS=('roas', 'mean'),
    平均CPA=('cpa', 'mean')
).round(4)

type_summary

In [ ]:
# 广告类型 × 渠道 CTR 均值热力图
pivot_ctr = df.pivot_table(values='ctr', index='campaign_type', 
                           columns='channel_used', aggfunc='mean')

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_ctr, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5)
plt.title('广告类型 × 渠道 平均CTR 热力图', fontsize=14)
plt.xlabel('投放渠道')
plt.ylabel('广告类型')
plt.savefig('eda_ctr_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 时间趋势分析

In [ ]:
# 按月汇总趋势
monthly = df.groupby(df['date'].dt.to_period('M')).agg(
    总花费=('spend', 'sum'),
    平均CTR=('ctr', 'mean'),
    平均ROAS=('roas', 'mean'),
    总转化=('conversions', 'sum')
).reset_index()
monthly['date'] = monthly['date'].dt.to_timestamp()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

metrics = [('总花费', '月度总花费趋势', 'steelblue'),
           ('平均CTR', '月度平均CTR趋势', 'coral'),
           ('平均ROAS', '月度平均ROAS趋势', 'seagreen'),
           ('总转化', '月度总转化量趋势', 'mediumpurple')]

for ax, (col, title, color) in zip(axes.flat, metrics):
    ax.plot(monthly['date'], monthly[col], marker='o', color=color, linewidth=2)
    ax.set_title(title, fontsize=13)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('eda_monthly_trends.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 花费 vs ROAS 散点图

In [ ]:
plt.figure(figsize=(12, 7))
scatter = plt.scatter(df['spend'], df['roas'], 
                      c=df['channel_used'].astype('category').cat.codes,
                      s=df['conversions'] * 2, alpha=0.4, cmap='Set1')

# 图例
channels = df['channel_used'].unique()
handles = [plt.scatter([], [], c=plt.cm.Set1(i / len(channels)), s=60, label=ch) 
           for i, ch in enumerate(channels)]
plt.legend(handles=handles, title='渠道', loc='upper right')

plt.xlabel('广告花费 ($)', fontsize=12)
plt.ylabel('ROAS', fontsize=12)
plt.title('花费 vs ROAS（气泡大小 = 转化量）', fontsize=14)
plt.savefig('eda_spend_vs_roas.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5 相关性热力图

In [ ]:
corr_cols = ['spend', 'impressions', 'clicks', 'conversions', 
             'ctr', 'cvr', 'cpc', 'cpm', 'roas', 'roi', 'engagement_score']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='coolwarm', center=0, linewidths=0.5, square=True)
plt.title('核心指标相关性热力图', fontsize=14)
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.6 渠道花费占比

In [ ]:
channel_spend = df.groupby('channel_used')['spend'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 饼图
axes[0].pie(channel_spend, labels=channel_spend.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2'), startangle=90)
axes[0].set_title('各渠道花费占比', fontsize=14)

# 条形图
axes[1].barh(channel_spend.index, channel_spend.values, color=sns.color_palette('Set2'))
axes[1].set_xlabel('总花费 ($)')
axes[1].set_title('各渠道总花费', fontsize=14)
for i, v in enumerate(channel_spend.values):
    axes[1].text(v + 1000, i, f'${v:,.0f}', va='center')

plt.tight_layout()
plt.savefig('eda_channel_spend.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 导出清洗后数据

In [ ]:
# 选取Tableau需要的列并导出
export_cols = [
    'campaign_id', 'company', 'campaign_type', 'target_audience',
    'duration', 'channel_used', 'location', 'language', 'customer_segment',
    'date', 'year', 'month', 'weekday',
    'spend', 'impressions', 'clicks', 'conversions',
    'ctr', 'cvr', 'cpc', 'cpm', 'roas', 'roi', 'cpa',
    'revenue', 'engagement_score'
]

df_export = df[export_cols].copy()
df_export.to_csv('cleaned_campaign_data.csv', index=False)

print(f'导出完成: cleaned_campaign_data.csv')
print(f'形状: {df_export.shape}')
print(f'\n--- 导出数据前5行 ---')
df_export.head()

In [ ]:
# 数据质量总结
print('=' * 50)
print('数据准备完成 — 总结')
print('=' * 50)
print(f'原始数据: 200,000 行 × 16 列')
print(f'清洗后:   {df_export.shape[0]:,} 行 × {df_export.shape[1]} 列')
print(f'日期范围: {df["date"].min().date()} ~ {df["date"].max().date()}')
print(f'渠道数量: {df["channel_used"].nunique()} ({list(df["channel_used"].unique())})')
print(f'广告类型: {df["campaign_type"].nunique()} ({list(df["campaign_type"].unique())})')
print(f'新增计算字段: CTR, CVR, CPC, CPM, ROAS, CPA, Revenue, Conversions')
print(f'\n输出文件: cleaned_campaign_data.csv → 供Tableau Dashboard使用')